In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install gymnasium sb3-contrib stable-baselines3 --quiet

In [ ]:
# ── 2. System path setup ─────────────────────────────────────────────────────
import sys, os
import numpy as np

DATASET_PATH = "/kaggle/input/datasets/nguyennhatphong/roguelike-rl-env"
sys.path.insert(0, DATASET_PATH)

print("Python version:", sys.version)
print("Dataset files:", os.listdir(DATASET_PATH)[:10])

In [ ]:
# ── 3. Balanced Environment Wrapper ──────────────────────────────────────────
from python_roguelike.env.roguelike_env import RoguelikeEnv
from python_roguelike.data.enums import GameState
from sb3_contrib.common.wrappers import ActionMasker


class BalancedEnv(RoguelikeEnv):
    """
    Balanced agent reward shaping:
    - Equal weight on HP gain and HP loss (symmetric)
    - Gold rewards incentivize shopping wisely
    - Moderate step penalty balances speed vs safety
    - Win bonus includes HP retention (but less than Defensive)
    """

    def step(self, action: int):
        run = self._controller.current_run
        prev_hp = run.the_hero.current_health
        prev_gold = run.the_hero.current_gold
        prev_floor = run.current_floor

        obs, _base_reward, terminated, truncated, info = super().step(action)

        run = self._controller.current_run
        hero = run.the_hero
        reward = 0.0

        # HP — symmetric, moderate weight
        hp_delta = hero.current_health - prev_hp
        reward += hp_delta * 0.2

        # Gold — incentivize economy and smart shopping
        gold_delta = hero.current_gold - prev_gold
        if gold_delta > 0:
            reward += gold_delta * 0.02

        # Floor progress — meaningful reward
        floor_delta = run.current_floor - prev_floor
        if floor_delta > 0:
            reward += floor_delta * 8.0

        # Moderate step cost — don't waste time, but survival matters
        reward -= 0.01

        # Terminal rewards — balanced win/loss
        if terminated:
            floors_cleared = run.current_floor + 1
            if hero.current_health > 0:
                hp_pct = hero.current_health / hero.max_health
                reward += 200.0 + floors_cleared * 10.0 + hp_pct * 50.0
            else:
                reward -= 75.0

        return obs, reward, terminated, truncated, info


json_path = os.path.join(DATASET_PATH, "python_roguelike", "GameData.json")
env = BalancedEnv(seed=42, json_path=json_path)
obs, info = env.reset()
print("   BalancedEnv created — obs shape:", obs.shape)
print("   Action space:", env.action_space)

In [ ]:
# ── 4. Validate Action Masking ───────────────────────────────────────────────
masked_env = ActionMasker(env, lambda e: e.action_masks())
obs, info = masked_env.reset()

for step in range(200):
    mask = masked_env.action_masks()
    valid = np.where(mask)[0]
    obs, reward, terminated, truncated, info = masked_env.step(int(np.random.choice(valid)))
    if terminated or truncated:
        obs, info = masked_env.reset()

print("   Masking validation passed")

In [ ]:
# ── 5. Vectorized Environment ─────────────────────────────────────────────────
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor

N_ENVS = 4

def make_env(seed_offset: int):
    def _init():
        _env = BalancedEnv(seed=3000 + seed_offset, json_path=json_path, max_steps=10_000)
        return ActionMasker(_env, lambda e: e.action_masks())
    return _init

vec_env = VecMonitor(SubprocVecEnv([make_env(i) for i in range(N_ENVS)]))
print(f"   VecEnv ready: {N_ENVS} parallel envs")

In [ ]:
# ── 6. MaskablePPO Model ─────────────────────────────────────────────────────
from sb3_contrib import MaskablePPO

policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256]))

model = MaskablePPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    n_steps=2048,
    batch_size=512,
    n_epochs=10,
    learning_rate=3e-4,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    tensorboard_log="/kaggle/working/tensorboard/balanced",
)
print("Model created. Policy params:", sum(p.numel() for p in model.policy.parameters()))

In [ ]:
# ── 7. Callbacks ──────────────────────────────────────────────────────────────
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, CallbackList

eval_env = ActionMasker(BalancedEnv(seed=7777, json_path=json_path), lambda e: e.action_masks())

callbacks = CallbackList([
    CheckpointCallback(
        save_freq=50_000,
        save_path="/kaggle/working/checkpoints/balanced",
        name_prefix="balanced_ppo",
        verbose=1,
    ),
    EvalCallback(
        eval_env,
        best_model_save_path="/kaggle/working/best_model/balanced",
        log_path="/kaggle/working/eval_logs/balanced",
        eval_freq=25_000,
        n_eval_episodes=10,
        deterministic=True,
        verbose=1,
    ),
])
print("   Callbacks configured")

In [ ]:
# ── 8. Training ──────────────────────────────────────────────────────────────
TOTAL_TIMESTEPS = 1_000_000

print(f"   Starting Balanced Agent training for {TOTAL_TIMESTEPS:,} timesteps...")
print(f"   Strategy: Equal offense/defense weighting")
print(f"   Symmetric HP reward ±0.2/HP | Floor reward +8 | Loss penalty -75")
print()

model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callbacks, progress_bar=True)

model.save("/kaggle/working/balanced_ppo_final")
print("\n   Training complete! Model saved to: /kaggle/working/balanced_ppo_final")

In [ ]:
# ── 9. Evaluation ─────────────────────────────────────────────────────────────
model = MaskablePPO.load("/kaggle/working/best_model/balanced/best_model")

N_EVAL = 50
floors, wins, hp_pcts, total_rewards = [], 0, [], []

for ep in range(N_EVAL):
    ep_env = ActionMasker(BalancedEnv(seed=ep, json_path=json_path), lambda e: e.action_masks())
    obs, info = ep_env.reset()
    done = False
    ep_reward = 0.0

    while not done:
        action, _ = model.predict(obs, deterministic=True, action_masks=ep_env.action_masks())
        obs, reward, terminated, truncated, info = ep_env.step(int(action))
        ep_reward += reward
        done = terminated or truncated

    floors.append(info["floor"])
    total_rewards.append(ep_reward)
    if info["hp"] > 0 and terminated:
        wins += 1
        hp_pcts.append(info["hp"] / info["max_hp"])

print(f"\n Balanced Agent Evaluation Results ({N_EVAL} episodes)")
print(f"═" * 50)
print(f"  Win Rate:          {wins/N_EVAL*100:.1f}%  ({wins}/{N_EVAL})")
print(f"  Avg Floor Reached: {np.mean(floors):.1f} / 15")
print(f"  Avg Episode Reward:{np.mean(total_rewards):.1f}")
if hp_pcts:
    print(f"  Avg HP% on Win:    {np.mean(hp_pcts)*100:.1f}%")